# Chapter 16 &mdash; Co-NP and Co-NPC: Primes versus Composites

**Concept 11 of the Chapter 16 decomposition:** *Co-NP and Co-NPC: Primes versus Composites*

$L\in$ Co-NP iff $\overline{L}\in$ NP; Primes is in both, which was evidence it lies in $P$.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16/Concept-Co-NP-And-Primes/Concept-Co-NP-And-Primes.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


$$L \in \text{Co-NP} \iff \overline{L} \in NP$$

**Composites** is in NP: the certificate is a factor. So **Primes** is in Co-NP
&mdash; the certificate for "not prime" is that same factor.

Primes turned out to be in NP too (Pratt, 1975, via a certificate built from a
primitive root), so Primes $\in NP \cap$ Co-NP. That was taken as strong evidence it
lies in $P$, because a problem complete for NP is believed not to be in Co-NP. In 2002
**AKS** proved Primes $\in P$ outright.

The general intuition: **NP is about easy-to-verify yes-answers; Co-NP about
easy-to-verify no-answers.** A problem in both is unlikely to be NP-complete, and is a
good candidate for $P$.

## 2. Definitions

### Certificates in both directions

In [ ]:
def cert_composite(n):
    # certificate for "n is composite": a non-trivial factor
    for d in range(2, int(n ** 0.5) + 1):
        if n % d == 0: return d
    return None

def verify_composite(n, d):
    return isinstance(d, int) and 1 < d < n and n % d == 0

def is_prime(n):
    return n > 1 and cert_composite(n) is None

### Pratt-style certificate for primality

In [ ]:
def factorise(n):
    f, d = [], 2
    while d * d <= n:
        while n % d == 0: f.append(d); n //= d
        d += 1
    if n > 1: f.append(n)
    return sorted(set(f))

# NOTE: a Jove star-import shadows the builtin three-argument pow(), so
# modular exponentiation gets its own helper here.
def modpow(b, e, m):
    r, b = 1, b % m
    while e:
        if e & 1: r = (r * b) % m
        b = (b * b) % m
        e >>= 1
    return r

def pratt_certificate(p):
    # (p, a, [prime factors of p-1]) with a a primitive root mod p
    if p == 2: return (2, 1, [])
    fs = factorise(p - 1)
    for a in range(2, p):
        if modpow(a, p - 1, p) != 1: continue
        if all(modpow(a, (p - 1) // q, p) != 1 for q in fs):
            return (p, a, fs)
    return None

def verify_pratt(cert):
    p, a, fs = cert
    if p == 2: return True
    if modpow(a, p - 1, p) != 1: return False
    return all(modpow(a, (p - 1) // q, p) != 1 for q in fs)

## 3. Tests

**Composites is in NP:** the certificate is a factor.

In [ ]:
for n in [15, 91, 1001, 7919]:
    d = cert_composite(n)
    print("  %-6d composite? %-6s certificate %s verifies? %s"
          % (n, d is not None, d, verify_composite(n, d) if d else '-'))
assert verify_composite(91, 7)
assert cert_composite(7919) is None

**So Primes is in Co-NP:** the same factor refutes primality.

In [ ]:
print("to show n is NOT prime : exhibit a factor -- short, fast to check")
print("to show n IS prime     : ??? -- no obvious short certificate")
print()
print("That asymmetry is exactly what Co-NP names.")

**Primes is also in NP**, by Pratt's recursive certificate.

In [ ]:
for p in [2, 7, 13, 97, 1009]:
    c_ = pratt_certificate(p)
    print("  p=%-6d certificate %-28s verifies? %s"
          % (p, str(c_), verify_pratt(c_)))
    assert c_ and verify_pratt(c_)

The certificate fails for composites &mdash; so it really certifies primality.

In [ ]:
for n in [15, 21, 91]:
    c_ = pratt_certificate(n)
    print("  n=%-4d (composite) certificate : %s" % (n, c_))
    assert c_ is None

So Primes $\in NP \cap$ Co-NP, which was the evidence.

In [ ]:
print("Primes in NP     : Pratt 1975")
print("Primes in Co-NP  : exhibit a factor")
print()
print("A language complete for NP is believed NOT to be in Co-NP, so a")
print("problem in both is probably not NP-complete -- and a good candidate")
print("for P.  AKS confirmed it in 2002.")

The symmetry, stated.

In [ ]:
print("%-12s %-34s %s" % ("class", "easy to verify", "example"))
print("%-12s %-34s %s" % ("NP", "YES answers", "composites, SAT"))
print("%-12s %-34s %s" % ("Co-NP", "NO answers", "primes, tautology"))
print("%-12s %-34s %s" % ("NP ^ Co-NP", "both", "primes, factoring (decision)"))
print()
print("P is inside the intersection.  Whether the intersection equals P")
print("is another open question.")

## 4. Exercises


1. Is TAUTOLOGY in NP? In Co-NP? Why the asymmetry with SAT?
2. Verify Pratt's certificate for $p=41$ by hand.
3. Is factoring (as a decision problem) known to be in $P$? In NP $\cap$ Co-NP?

In [ ]:
# Your work for the exercises above.